# **Regularization: how the weights of Ridge, Lasso and Elastic Net change as $\lambda$ grows**

### A tutorial for the lesson “Regularization: how to choose one solution out of many”

In the lesson the three penalties are worked out on a toy example with two identical features: Ridge splits the contribution between them equally and shrinks it smoothly toward zero, Lasso can zero the contribution at a finite $\lambda$, Elastic Net zeroes it like Lasso and splits it like Ridge. Here the same three models are trained on California Housing, with the same features and the same split as in the notebook `HW1_linear_models_weights_and_predictions_en`, and shown in two interactive Plotly charts:

1. the weights of all eight features against $\lambda$, with a Ridge / Lasso / Elastic Net switch;
2. the weights of the three models side by side at one $\lambda$ chosen with a slider.

There are no self-check quizzes on this notebook.

## The lesson's objectives and the sklearn parameters

In the lesson all three problems are written without dividing by the number of objects $n$:

$$\text{Ridge:}\ \|y-X\beta\|_2^2+\lambda\|\beta\|_2^2,\qquad \text{Lasso:}\ \|y-X\beta\|_2^2+\lambda\|\beta\|_1,$$

$$\text{Elastic Net:}\ \|y-X\beta\|_2^2+\lambda_1\|\beta\|_1+\lambda_2\|\beta\|_2^2.$$

In sklearn `Ridge` is written the same way, so `alpha` $=\lambda$. `Lasso` and `ElasticNet` divide the error by $2n$:

$$\text{Lasso:}\ \frac{1}{2n}\|y-X\beta\|_2^2+\alpha\|\beta\|_1,\qquad \text{ElasticNet:}\ \frac{1}{2n}\|y-X\beta\|_2^2+\alpha\rho\|\beta\|_1+\frac{\alpha(1-\rho)}{2}\|\beta\|_2^2,$$

where $\rho$ is `l1_ratio`. Dividing the lesson's objective by $2n$ gives `alpha` $=\lambda/(2n)$ for Lasso, and `alpha` $=\lambda_1/(2n)+\lambda_2/n$ with `l1_ratio` $=\dfrac{\lambda_1/(2n)}{\text{alpha}}$ for Elastic Net. Below $\lambda$ is always on the lesson's scale, and Elastic Net uses $\lambda_1=\lambda_2=\lambda$ (then `l1_ratio` $=1/3$ at any $\lambda$). sklearn does not penalize the intercept, just as in the lesson “Regularization: how to choose one solution out of many”.

## Data

California Housing, the split and the standardization are as in the notebook `HW1_linear_models_weights_and_predictions_en`: `random_state=42`, `test_size=0.25`, a `StandardScaler` fitted on `X_train`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet

RANDOM_STATE = 42

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.25)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

labels = list(X.columns)
n = X_train_scaled.shape[0]
print('n =', n)

## Weight paths

We train the three models on a grid of 51 values of $\lambda$ from $1$ to $10^5$ (evenly spaced on a log scale).

In [ ]:
lambdas = np.logspace(0, 5, 51)  # λ from 1 to 100000 on the lesson's scale

def fit_weights(lam):
    ridge = Ridge(alpha=lam).fit(X_train_scaled, y_train)
    lasso = Lasso(alpha=lam / (2 * n), max_iter=50000).fit(X_train_scaled, y_train)
    # Elastic Net with lambda_1 = lambda_2 = lam
    alpha_en = lam / (2 * n) + lam / n
    enet = ElasticNet(alpha=alpha_en, l1_ratio=(lam / (2 * n)) / alpha_en, max_iter=50000).fit(X_train_scaled, y_train)
    return ridge.coef_, lasso.coef_, enet.coef_

names = ['Ridge', 'Lasso', 'Elastic Net']
weights = {name: [] for name in names}
for lam in lambdas:
    for name, w in zip(names, fit_weights(lam)):
        weights[name].append(w)
paths = {name: pd.DataFrame(w, index=lambdas, columns=labels) for name, w in weights.items()}

# the weights at the smallest and at the largest lambda of the grid
pd.concat({name: p.iloc[[0, -1]].T for name, p in paths.items()}, axis=1).round(4)

### Chart 1. Feature weights against $\lambda$

The buttons above the chart switch the model. The $\lambda$ axis is logarithmic.

In [ ]:
colors = px.colors.qualitative.Plotly

fig = go.Figure()
for k, name in enumerate(names):
    for j, f in enumerate(labels):
        fig.add_trace(go.Scatter(x=lambdas, y=paths[name][f], mode='lines', name=f,
                                 line=dict(color=colors[j]), visible=(k == 0)))

buttons = [dict(label=name, method='update',
                args=[{'visible': [kk == k for kk in range(len(names)) for _ in labels]},
                      {'title': f'{name}: weights against λ'}])
           for k, name in enumerate(names)]
fig.update_layout(updatemenus=[dict(type='buttons', direction='right', buttons=buttons, x=0, y=1.18, xanchor='left')],
                  title='Ridge: weights against λ', height=520,
                  xaxis=dict(type='log', title='λ (lesson scale, log axis)'),
                  yaxis=dict(title='weight of a standardized feature'))
fig.show()

### Chart 2. The weights of the three models at one $\lambda$

The slider under the chart picks $\lambda$ from the same grid; all three models are shown side by side on one axis.

In [ ]:
fig2 = make_subplots(rows=1, cols=3, subplot_titles=names, shared_yaxes=True)
for k, name in enumerate(names):
    fig2.add_trace(go.Bar(x=labels, y=paths[name].iloc[0].tolist(), marker_color=colors[:len(labels)], showlegend=False),
                   row=1, col=k + 1)

steps = [dict(method='restyle', label=f'{lam:.0f}',
              args=[{'y': [paths[name].iloc[i].tolist() for name in names]}])
         for i, lam in enumerate(lambdas)]
ymax = max(p.abs().max().max() for p in paths.values()) * 1.1
fig2.update_yaxes(range=[-ymax, ymax])
fig2.update_yaxes(title_text='weight of a standardized feature', row=1, col=1)
fig2.update_xaxes(tickangle=45)
fig2.update_layout(title='Weights of the three models at one λ (move the slider)', height=560,
                   sliders=[dict(active=0, currentvalue=dict(prefix='λ = '), steps=steps, pad=dict(t=90))])
fig2.show()

## From which $\lambda$ a weight becomes zero

For every feature, the smallest $\lambda$ of the grid from which the weight stays zero to the end of the grid. `NaN` means the weight did not become zero at any $\lambda$ of the grid.

In [ ]:
def first_zero(path):
    # the smallest lambda of the grid from which the feature weight stays zero to the end of the grid
    zero = (path.abs() < 1e-10).values
    tail = np.flip(np.cumprod(np.flip(zero, axis=0), axis=0), axis=0).astype(bool)
    return pd.Series([path.index[tail[:, j].argmax()] if tail[:, j].any() else np.nan for j in range(path.shape[1])],
                     index=path.columns)

zero_table = pd.DataFrame({name: first_zero(p) for name, p in paths.items()})
zero_table.sort_values('Lasso').round(0)

The weights of Latitude and Longitude — a pair of features with a correlation of $-0.92$ on `X_train` — in the three models, every fifth value of the grid:

In [ ]:
lat_lon = pd.concat({name: p[['Latitude', 'Longitude']] for name, p in paths.items()}, axis=1)
lat_lon.iloc[::5].round(4)

## What the charts and the tables show

- **Ridge** zeroes no weight on the whole grid: all the weights shrink smoothly, and even at $\lambda=10^5$ the largest absolute weight belongs to MedInc, $0.1057$. Shrinking need not keep the sign: the weight of AveRooms is $-0.3019$ at $\lambda=1$ and $+0.0212$ at $\lambda=10^5$.
- **Lasso** removes the features one by one: Population from $\lambda\approx50$, AveRooms from $\lambda\approx794$, AveBedrms and AveOccup from $\lambda\approx1585$, Longitude from $\lambda\approx2512$, Latitude from $\lambda\approx3981$, HouseAge from $\lambda\approx6310$, and MedInc last, from $\lambda\approx25119$. Of the pair Latitude and Longitude, Lasso zeroes Longitude first: at $\lambda\approx3162$ the weight of Latitude is still $-0.0084$ while Longitude is already $0$.
- **Elastic Net** zeroes the weights like Lasso, and in almost the same order; only Population ($\lambda\approx25$) and AveBedrms ($\lambda\approx1000$) go earlier. Before zeroing it shrinks harder than both other models, because both penalties act at once: at $\lambda=1000$ the weights of Latitude and Longitude are $-0.2850$ and $-0.2437$ for Elastic Net, $-0.4950$ and $-0.4611$ for Ridge, and $-0.5120$ and $-0.4701$ for Lasso.

The link with the notebook `HW1_linear_models_weights_and_predictions_en`: there Lasso was trained with `alpha` from an sklearn grid, and `alpha=0.1` on the lesson's scale is $\lambda=2n\cdot0.1=3096$. This value lies between $\lambda\approx2512$, from which Lasso zeroes Longitude, and $\lambda\approx3981$, from which it zeroes Latitude — which is why in that notebook at `alpha=0.1` Longitude has a zero weight and Latitude does not yet.